In [ ]:
# Copyright Aditya Rane
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Session 3: Window Functions + Top Interview Patterns
> This notebook introduces Session 3: Window Functions + Top Interview Patterns

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/Adi8885/python-for-beginners/blob/main/tutorial-May2026-InterviewPrep-Batch/Session1%20-%20SQL%20Mastery%20—%20The%20Foundations.ipynb">
      <img src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Run in Colab
    </a>
  </td>
    <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https://raw.githubusercontent.com/Adi8885/python-for-beginners/refs/heads/main/tutorial-May2026-InterviewPrep-Batch/Session1%20-%20SQL%20Mastery%20—%20The%20Foundations.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Run in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/Adi8885/python-for-beginners/blob/main/tutorial-May2026-InterviewPrep-Batch/Session1%20-%20SQL%20Mastery%20—%20The%20Foundations.ipynb">
      <img src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/Adi8885/python-for-beginners/refs/heads/main/tutorial-May2026-InterviewPrep-Batch/Session1%20-%20SQL%20Mastery%20—%20The%20Foundations.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
</table>

# 🎓 Session 3: Window Functions & Advanced SQL Patterns

| Item | Details |
| :--- | :--- |
| ⏳ **Duration** | 90–120 minutes |
| 🎯 **Level** | Intermediate |
| 🚀 **Main Goal** | Students should be able to solve ranking, latest record, running total, and comparison-based SQL problems. |

---

## 🌟 Why This Session?
In SQL interviews and data analytics roles, **Window Functions** are the most highly-tested topic! Why? Because they allow you to perform advanced calculations (like running totals, moving averages, and ranks) while keeping the original rows intact. They turn complex self-joins into single, elegant lines of SQL. 💼✨

---

## 🧩 Part 1: Why Window Functions Are Needed — 10 mins

Let's understand the difference between `GROUP BY` and window functions.

### 1. `GROUP BY` (The Collapser 🧹)
`GROUP BY` collapses multiple rows into a single summary row. Once grouped, you lose the individual row details.

**Query:**
```sql
SELECT customer_id, SUM(amount) AS total_amount
FROM orders
GROUP BY customer_id;
```

**Visual Output:**
| customer_id | total_amount |
| :--- | :--- |
| `A` | `30` |
| `B` | `15` |

---

### 2. Window Function (The Calculator 🧮)
A **Window Function** performs calculations across a set of rows (a "window") related to the current row, but **retains all individual rows** and appends the result as a new column.

**Query:**
```sql
SELECT
  customer_id,
  order_id,
  amount,
  SUM(amount) OVER(PARTITION BY customer_id) AS customer_total
FROM orders;
```

**Visual Output:**
| customer_id | order_id | amount | customer_total |
| :--- | :--- | :--- | :--- |
| `A` | `1` | `10` | **`30`** |
| `A` | `2` | `20` | **`30`** |
| `B` | `3` | `15` | **`15`** |

### 📊 Visualizing the Difference

```mermaid
graph TD
    subgraph GroupBy [GROUP BY - Collapses Rows]
        A1["Row 1: Customer A ($10)"] --> G1["Group Customer A"]
        A2["Row 2: Customer A ($20)"] --> G1
        A3["Row 3: Customer B ($15)"] --> G2["Group Customer B"]
        G1 --> Out1["Output 1: Customer A, $30"]
        G2 --> Out2["Output 2: Customer B, $15"]
    end
    
    subgraph WindowFunc [Window Function - Retains Rows]
        W1["Row 1: Customer A ($10)"] --> Win1["Output 1: Customer A, $10, Total: $30"]
        W2["Row 2: Customer A ($20)"] --> Win2["Output 2: Customer A, $20, Total: $30"]
        W3["Row 3: Customer B ($15)"] --> Win3["Output 3: Customer B, $15, Total: $15"]
    end
```

---

## 🏗️ Part 2: The Core Anatomy of a Window Function — 15 mins

To invoke a window function, we use the `OVER()` clause. The basic structure is:

```sql
FUNCTION() OVER (
    PARTITION BY partition_column
    ORDER BY sort_column
    ROWS/RANGE frame_clause
)
```

- **`PARTITION BY`** 🍰: Slices your data into groups (or partitions). The window calculation restarts for each group.
- **`ORDER BY`** 📶: Sorts the rows *within* each partition. Essential for ranking and running totals.
- **`ROWS/RANGE`** 🖼️: Limits the rows within the partition used for the calculation (e.g. "preceding 3 rows").

---

## 👑 Part 3: Ranking Problems (`ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`) — 25 mins

Understanding the differences between these ranking functions is a classic interview question.

### 📝 The Ranking Comparison Table

| Function | Behavior | Tie Handling | Example Sequence (1st, 1st, 3rd) |
| :--- | :--- | :--- | :--- |
| **`ROW_NUMBER()`** | Sequential integers starting at 1. | Assigns arbitrary order for ties. | `1, 2, 3` |
| **`RANK()`** | Rank with gaps after ties. | Tied rows get the same rank; next rank is skipped. | `1, 1, 3` |
| **`DENSE_RANK()`** | Rank without gaps after ties. | Tied rows get the same rank; next rank is sequential. | `1, 1, 2` |

### 🎬 IMDb Example: Ranking Marathi Movies by Rating
Let's run a query to rank Marathi movies.

```sql
SELECT l.Name AS Language,
       m.title AS Movie_Title,
       m.rating AS Rating,
       ROW_NUMBER() OVER(PARTITION BY l.Name ORDER BY m.rating DESC) AS Row_Num,
       RANK() OVER(PARTITION BY l.Name ORDER BY m.rating DESC) AS Rank_Num,
       DENSE_RANK() OVER(PARTITION BY l.Name ORDER BY m.rating DESC) AS Dense_Rank_Num
FROM Movie m
INNER JOIN M_Language ml ON m.MID = ml.MID
INNER JOIN Language l ON ml.LAID = l.LAID
WHERE l.Name = 'Marathi'
ORDER BY m.rating DESC;
```

**Results:**
| Language | Movie_Title | Rating | Row_Num | Rank_Num | Dense_Rank_Num |
| :--- | :--- | :--- | :--- | :--- | :--- |
| Marathi | Tumbbad | 8.5 | 1 | 1 | 1 |
| Marathi | Ek Hota Vidushak | 7.9 | 2 | 2 | 2 |
| Marathi | Kaksparsh | 7.8 | 3 | 3 | 3 |
| Marathi | Kairee | 7.7 | 4 | **4** | **4** |
| Marathi | Court | 7.7 | 5 | **4** | **4** |
| Marathi | Pinjra | 7.6 | 6 | **6** *(skips 5)* | **5** *(dense)* |

---

## 🎯 Part 4: Latest Record Pattern (Filtering Rankings) — 20 mins

**Goal:** Find the highest-rated movie for each language.

### ⚠️ The Execution Order Trap!
Why can't you do this?
```sql
-- ❌ ERROR!
SELECT title, ROW_NUMBER() OVER(PARTITION BY LAID ORDER BY rating DESC) AS rn
FROM Movie
WHERE rn = 1;
```
Because SQL's logical execution order runs the `WHERE` clause **before** the `SELECT` clause (where window functions are calculated). Therefore, `rn` does not exist when `WHERE` runs.

### 🔑 The Fix: Use a CTE (Common Table Expression)
```sql
WITH RankedMovies AS (
    SELECT l.Name AS Language_Name,
           m.title AS Movie_Title,
           m.rating AS Rating,
           ROW_NUMBER() OVER(PARTITION BY l.Name ORDER BY m.rating DESC) AS rn
    FROM Movie m
    INNER JOIN M_Language ml ON m.MID = ml.MID
    INNER JOIN Language l ON ml.LAID = l.LAID
)
SELECT Language_Name, Movie_Title, Rating
FROM RankedMovies
WHERE rn = 1
ORDER BY Rating DESC;
```

---

## 📈 Part 5: Running Totals (Cumulative Sums) — 20 mins

A running total is a cumulative sum of data. It is calculated by using `SUM()` with `ORDER BY` inside `OVER()`.

### 🎬 IMDb Example: Running Total of Movies Released (2010–2018)
```sql
SELECT year AS Release_Year,
       COUNT(*) AS Movies_Released,
       SUM(COUNT(*)) OVER(ORDER BY year) AS Cumulative_Movies
FROM Movie
WHERE year >= '2010' AND year <= '2018'
GROUP BY year
ORDER BY year;
```

**Results:**
| Release_Year | Movies_Released | Cumulative_Movies |
| :--- | :--- | :--- |
| `2010` | `117` | `117` |
| `2011` | `109` | `226` *(117 + 109)* |
| `2012` | `108` | `334` *(226 + 108)* |
| `2013` | `127` | `461` |
| `2014` | `118` | `579` |
| `2015` | `109` | `688` |
| `2016` | `118` | `806` |
| `2017` | `118` | `924` |
| `2018` | `93` | `1017` |

---

## 🔄 Part 6: Comparison-Based Problems (`LAG()` and `LEAD()`) — 25 mins

Sometimes you need to compare a value in the current row to a value in a previous or subsequent row.
- **`LAG(column, offset)`**: Looks backward (returns value from `offset` rows before).
- **`LEAD(column, offset)`**: Looks forward (returns value from `offset` rows after).

### 🎬 IMDb Example: YoY Change in USA Movies
```sql
WITH USA_Annual_Movies AS (
    SELECT year AS Release_Year,
           COUNT(*) AS Movie_Count
    FROM Movie m
    INNER JOIN M_Country mc ON m.MID = mc.MID
    INNER JOIN Country c ON mc.CID = c.CID
    WHERE c.Name = 'USA' AND year >= '2010' AND year <= '2018'
    GROUP BY year
)
SELECT Release_Year,
       Movie_Count,
       LAG(Movie_Count, 1) OVER(ORDER BY Release_Year) AS Prev_Year_Count,
       Movie_Count - LAG(Movie_Count, 1) OVER(ORDER BY Release_Year) AS YoY_Change
FROM USA_Annual_Movies
ORDER BY Release_Year;
```

**Results:**
| Release_Year | Movie_Count | Prev_Year_Count | YoY_Change |
| :--- | :--- | :--- | :--- |
| `2010` | `2` | `NULL` | `NULL` |
| `2011` | `2` | `2` | `0` |
| `2012` | `4` | `2` | `+2` |
| `2013` | `2` | `4` | `-2` |
| `2014` | `6` | `2` | `+4` |
| `2015` | `1` | `6` | `-5` |
| `2016` | `3` | `1` | `+2` |
| `2017` | `1` | `3` | `-2` |
| `2018` | `1` | `1` | `0` |

---

## 🏠 Session 3 Homework: The Window Function Challenge

Test your skills using the `Db-IMDB-SQL-Dataset.db`!

### 📝 The Challenge Set

| No. | Problem Statement | Difficulty | Hint 💡 |
| :--- | :--- | :--- | :--- |
| **1** | For the year `'2018'`, rank all movies using `DENSE_RANK()` based on their rating. Show the highest-rated movies first. | ⭐⭐ | `DENSE_RANK() OVER(ORDER BY rating DESC)` |
| **2** | Find the highest-rated movie for each year. If there is a tie, display the movie with more votes. Use a CTE. | ⭐⭐⭐ | CTE + `ROW_NUMBER() OVER(PARTITION BY year ORDER BY rating DESC, num_votes DESC)` |
| **3** | Calculate the running total of movie votes per year for movies released from `'2015'` to `'2018'`. | ⭐⭐⭐ | Group by year first, then do `SUM(Year_Votes) OVER(ORDER BY year)` |
| **4** | For movies directed by `'Rajkumar Hirani'`, list the title, release year, rating, and the rating of his previous movie (by year) using `LAG()`. Also calculate the difference in rating from his previous movie. | ⭐⭐⭐⭐ | Join `Movie` ↔ `M_Director` ↔ `Person` and use `LAG(rating) OVER(ORDER BY year)` |

**Happy Querying!** 🚀💻

---

### 🔑 Answer Key for Instructors
(Refer to this section to verify student submissions or check answers!)

<details>
<summary><b>Click to show Answer Key</b></summary>

#### Query 1:
```sql
SELECT title,
       rating,
       DENSE_RANK() OVER (ORDER BY rating DESC) AS Movie_Rank
FROM Movie
WHERE year = '2018'
ORDER BY Movie_Rank;
```

#### Query 2:
```sql
WITH RankedMovies AS (
    SELECT year,
           title,
           rating,
           num_votes,
           ROW_NUMBER() OVER (PARTITION BY year ORDER BY rating DESC, num_votes DESC) AS rn
    FROM Movie
)
SELECT year, title, rating, num_votes
FROM RankedMovies
WHERE rn = 1
ORDER BY year DESC;
```

#### Query 3:
```sql
WITH AnnualVotes AS (
    SELECT year,
           SUM(num_votes) AS Year_Votes
    FROM Movie
    WHERE year IN ('2015', '2016', '2017', '2018')
    GROUP BY year
)
SELECT year,
       Year_Votes,
       SUM(Year_Votes) OVER (ORDER BY year) AS Running_Total_Votes
FROM AnnualVotes
ORDER BY year;
```

#### Query 4:
```sql
SELECT m.title AS Movie_Title,
       m.year AS Release_Year,
       m.rating AS Rating,
       LAG(m.rating) OVER (ORDER BY m.year) AS Prev_Movie_Rating,
       ROUND(m.rating - LAG(m.rating) OVER (ORDER BY m.year), 2) AS Rating_Diff
FROM Movie m
INNER JOIN M_Director md ON m.MID = md.MID
INNER JOIN Person p ON md.PID = p.PID
WHERE TRIM(p.Name) = 'Rajkumar Hirani'
ORDER BY m.year;
```
</details>
